# 204 — Postprocess S2 Tiles

Applies cultivated land suppression per tile.
No morphological ops at 10m resolution.
Run after `203_predict_sat.ipynb`.

In [ ]:
from config_s2 import *
from helpers_s2 import *
import time, gc
import numpy as np

print(f"Config loaded ✓  |  Thresholds: {PIXEL_THRESHOLDS}")

In [ ]:
# ── Load cultivated land ─────────────────────────────────────────────────────
cultivated_gdf = gpd.read_file(CULTIVATED_PATH)
cultivated_gdf["geometry"] = cultivated_gdf.geometry.make_valid()
print(f"Cultivated polygons: {len(cultivated_gdf)}")

In [ ]:
# ── Build tile list ───────────────────────────────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

ref_thresh = PIXEL_THRESHOLDS[0]
bin_dir    = s2_binary_tile_dir(ref_thresh)
bin_files  = sorted(f for f in os.listdir(bin_dir) if f.endswith(".tif"))
print(f"  Binary tiles: {len(bin_files)}")

# Only process tiles with lupine
to_process = []
for fname in bin_files:
    with rasterio.open(os.path.join(bin_dir, fname)) as src:
        data = src.read(1)
    if int((data == 1).sum()) >= 1:
        source_fname = fname.replace(f"_binary_t{ref_thresh}.tif", ".tif")
        to_process.append(source_fname)

# Resume
pp_existing = set(os.listdir(s2_pp_tile_dir(ref_thresh)))
remaining   = [f for f in to_process
               if f.replace(".tif", f"_pp_t{ref_thresh}.tif") not in pp_existing]

print(f"  With lupine:  {len(to_process)}")
print(f"  Done:         {len(to_process) - len(remaining)}")
print(f"  Remaining:    {len(remaining)}")

In [ ]:
# ── Postprocess per tile — cultivated suppression only ───────────────────────
print("── Postprocessing tiles ────────────────────────────────────")

t0 = time.time()

for i, source_fname in enumerate(remaining):
    remaining_count = len(remaining) - i - 1

    for thresh in PIXEL_THRESHOLDS:
        bin_fname = source_fname.replace(".tif", f"_binary_t{thresh}.tif")
        bin_path  = os.path.join(s2_binary_tile_dir(thresh), bin_fname)
        pp_fname  = source_fname.replace(".tif", f"_pp_t{thresh}.tif")
        pp_path   = os.path.join(s2_pp_tile_dir(thresh), pp_fname)

        if not os.path.exists(bin_path):
            continue

        with rasterio.open(bin_path) as src:
            binary  = src.read(1).astype(np.uint8)
            profile = src.profile.copy()

        # Cultivated suppression only
        binary = postprocess_tile_s2(binary, profile, cultivated_gdf, cult_buffer_m=CULT_BUFFER_M)

        if int((binary == 1).sum()) == 0:
            del binary; gc.collect()
            continue

        pp_profile = profile.copy()
        pp_profile.update(count=1, dtype="uint8", compress="lzw",
                          tiled=True, blockxsize=256, blockysize=256, nodata=None)
        with rasterio.open(pp_path, "w", **pp_profile) as dst:
            dst.write(binary[np.newaxis, :, :])
        del binary; gc.collect()

    elapsed = time.time() - t0
    done    = i + 1
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h\n")
    else:
        print(f"  [{i+1}/{len(remaining)}] {source_fname} — {remaining_count} left — ETA {eta/60:.1f}min")

print(f"\nDone. Time: {(time.time()-t0)/3600:.1f}h")
for thresh in PIXEL_THRESHOLDS:
    n = len(os.listdir(s2_pp_tile_dir(thresh)))
    print(f"  PP tiles t={thresh}: {n}")